# Textual Data Cleaning PipeLine

This is a code Pipeline specifcally tailored to the processing of textual data and informations and the pre-tokenisation phases, to ensure that we are training the model on accurate and properly treated textual corpa. 

## Packages

In [56]:
# I use this at the end of the pipeline when I am converting everything and loading a JSONL into
# a dataframe for further analysis down the line. Also becuase I am used by principle to have this in every line of code that I write. 
import pandas as pd
# Just gives me the array opps that I need in the main filtering stage. 
import numpy as np
# The package I use when I have to upload my data locally, for own re run ensure that when I upload dat from
# own computer to change the file paths since they won't be successful in re-runs. 
from pathlib import Path
# I use this package when i need to parse command-line arguments that control which pipeline stages are acc running and their
# respective configuration. 
import argparse
# Used in the deduplication stage generating SHA-256 hashes of normalized text content to identify exact duplicate documents. 
import hashlib
# need it for the output stage
import json
# lets me search and change the different strings using complex pattern descriptions. 
import re
# for error handling and debug output, particularly in the extraction stage when document parsing fails
import sys
# when I do the Unicode normalisation to ensure that all my characters are standardised 
import unicodedata
# Used in the extraction stage for counting repeated lines when detecting running headers/footers in PDF documents
from collections import Counter
# Used at the configuration and data structure stage for defining the Config and Doc classes that hold pipeline parameters and document metadata
from dataclasses import dataclass, field, asdict
# Used in the acquisition stage (Stage 1) for parsing date information from filenames and formatting them as YYYY-MM-DD
from datetime import datetime
# Used throughout the codebase for type hints, particularly in function signatures across all stages.
from typing import Optional
# Used in the extraction stage (Stage 1) for parsing HTML documents and extracting the main content while removing boilerplate tags like script, style, nav, and footer
from bs4 import BeautifulSoup     
# repairing mojibake (character encoding corruption) before applying other normalization steps
import ftfy
# or building MinHash signatures and LSH (Locality-Sensitive Hashing) indexes to identify near-duplicate documents that share significant textual overlap. 
from datasketch import MinHash, MinHashLSH     


## Importing the CSV

### Important for run through on local computer you must change the path to the correct one!!

In [57]:
# !! Path Name and File Path -- FED FOMC minutes and statements post scrape and in need of cleaning
FOMC_unclean_data_csv_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/FED BRANCH/POLICY DECISIONS/FOMC ANNOUNCMENTS")
FOMC_unclean_data_csv = pd.read_csv(FOMC_unclean_data_csv_path/ "fomc_minutes_statements.csv")
FOMC_unclean_data_csv.head(10)

,Date,Release Date,Type,Text
0,2026-06-17,2026-06-17,Statement,The Federal Open Market Committee approved the...
1,2026-04-29,2026-04-29,Statement,Recent indicators suggest that economic activi...
2,2026-04-29,2026-05-20,Minute,Minutes of the Federal Open Market Committee\n...
3,2026-03-18,2026-03-18,Statement,Available indicators suggest that economic act...
4,2026-03-18,2026-04-08,Minute,Minutes of the Federal Open Market Committee\n...
5,2026-01-28,2026-01-28,Statement,Available indicators suggest that economic act...
6,2026-01-28,2026-02-18,Minute,Minutes of the Federal Open Market Committee\n...
7,2025-12-10,2025-12-10,Statement,Available indicators suggest that economic act...
8,2025-12-10,2025-12-30,Minute,Minutes of the Federal Open Market Committee\n...
9,2025-10-29,2025-10-29,Statement,Available indicators suggest that economic act...


`Splitting the Data frame between minutes and also Statements

In [58]:
# Splitting is done below, using the pandas package. 
type_col = 'Type'
types = FOMC_unclean_data_csv[type_col].astype(str).str.strip()

# boolean masks (word-boundary to avoid partial matches)
is_statement = types.str.lower().str.contains(r'\bstatement\b', na=False)
is_minute = types.str.lower().str.contains(r'\bminute\b', na=False)
# split
statements_df = FOMC_unclean_data_csv[is_statement].copy()
minutes_df = FOMC_unclean_data_csv[is_minute].copy()

setup + stack statements and minutes into one frame

In [59]:
text_col = None   

# a simple function to detect wether there is text in a frame 
def _detect_text_col(df, exclude=()):
    cand = [c for c in df.columns if c not in exclude
            and not pd.api.types.is_numeric_dtype(df[c])
            and not pd.api.types.is_datetime64_any_dtype(df[c])]
    L = {c: df[c].dropna().astype(str).str.len().mean() for c in cand}
    return max(L, key=L.get) if L else None

if text_col is None:
    text_col = _detect_text_col(statements_df, exclude={type_col})

docs = pd.concat([statements_df.assign(doc_type="statement"), minutes_df.assign(doc_type="minutes")], ignore_index=True)
docs = docs[[text_col, "doc_type"]].rename(columns={text_col: "raw_text"})
print("text column:", text_col, "| documents:", docs.shape)
docs.head()

text column: Text | documents: (465, 2)


,raw_text,doc_type
0,The Federal Open Market Committee approved the...,statement
1,Recent indicators suggest that economic activi...,statement
2,Available indicators suggest that economic act...,statement
3,Available indicators suggest that economic act...,statement
4,Available indicators suggest that economic act...,statement


clean / normalize the text

split into sentences

In [60]:
"""
FOMC statement isolation + normalisation + rule-based filtration + splitting.

This keeps the ORIGINAL cleaning phases (voting-record cutoff, header/nav strip,
media/phone/email/URL/implementation-note noise, roster guard) and layers on:

  - clean_text          : unicode / quote / dash normalisation + de-hyphenation.
  - Data & Title Filter  : keep only sentences with a Panel A1 or B1 term (Table 1).
  - Sentence Splitting   : split at {but, however, even though, although, while, ;};
                           keep the split only if EVERY segment has a Table 1 term.

Order matters for "no jargon picked up": the wrapper is stripped FIRST, so a
voting line such as "Voting for the monetary policy action were ..." (which
contains the B1 term 'monetary policy') is removed before the keyword filter
ever sees it.

Assumes `docs` is a DataFrame with `raw_text` and `doc_type` columns. A base
sentence splitter `split` (your get_splitter() result) is used; a regex
fallback is provided if it isn't defined.
"""

import re
import unicodedata
import pandas as pd


# ===========================================================================
# 1. Normalisation
# ===========================================================================

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKC", s)
    s = (s.replace("\u201c", '"').replace("\u201d", '"').replace("\u2018", "'")
           .replace("\u2019", "'").replace("\u2013", "-").replace("\u2014", "-")
           .replace("\u00a0", " "))
    s = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", s)   # "infla-\ntion" -> "inflation"
    s = re.sub(r"\s*\n\s*", " ", s)                 # hard line wraps -> spaces
    s = re.sub(r"[ \t]+", " ", s).strip()
    return s


# Leading wrapper strip. Same header/nav tokens as HEADER_MARKERS below, but
# applied at the document level. clean_text collapses line breaks into spaces,
# which would otherwise glue "For immediate release / Share / Print / PDF / date"
# onto the first body sentence (so the per-sentence header skip would drop real
# content). Stripping it here keeps the first sentence intact. No-op when absent.
_MONTHS = (r"(?:january|february|march|april|may|june|july|august|september"
           r"|october|november|december)")
LEADING_WRAPPER = re.compile(
    r"^(?:\s*(?:"
    r"share|print|pdf"
    r"|for immediate release"
    r"|for release at[^\n]*?\d{4}"
    r"|last update[^\n]*?\d{4}"
    rf"|{_MONTHS}\s+\d{{1,2}},?\s+\d{{4}}"
    r"|\d{4}"
    r")\b[\s:,.\-]*)+",
    re.IGNORECASE,
)

def strip_leading_wrapper(s):
    return LEADING_WRAPPER.sub("", s).lstrip() if isinstance(s, str) else ""


# ===========================================================================
# 2. ORIGINAL cleaning patterns + functions (unchanged)
# ===========================================================================

# Marks the END of the statement body (voting/dissent paragraph onward).
END_MARKERS = re.compile(
    r'\bvoting (?:for|against|in favor)\b'
    r'|\bvoted (?:for|against|to|in favor|unanimously)\b'
    r'|voting on (?:this|the) (?:action|matter)',
    re.I,
)

# Leading release headers / scraped nav.
HEADER_MARKERS = re.compile(
    r'for immediate release|for release at|last update'
    r'|^\s*share\b|^\s*print\b|^\s*pdf\b',
    re.I,
)

# Trailing/standalone boilerplate (backup for docs with no voting marker).
NOISE = re.compile(
    r'for media inquiries|media inquiries|please (?:e-?mail|call)'
    r'|implementation note'
    r'|\bhttps?://|\bwww\.|federalreserve\.gov'
    r'|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'          # phone numbers
    r'|[\w.+-]+@[\w-]+\.[\w.]+'                  # email addresses
    r'|\[\s*email\s*protected\s*\]'             # scraper redaction artifact
    r'|board of governors of the federal reserve',
    re.I,
)

# Roster titles — only ever appear in the voting list, never in the body.
ROSTER = re.compile(
    r'\b(?:chair(?:man)?|vice chair(?:man)?|governor|secretary|president)\b',
    re.I,
)


def keep_statement_sentences(sentences):
    """Return only the statement body: skip leading headers, STOP at the voting
    record, drop stray boilerplate."""
    out = []
    for s in sentences:
        s = (s or "").strip()
        if not s:
            continue
        if END_MARKERS.search(s):      # voting/dissent block -> body is over
            break
        if HEADER_MARKERS.search(s):   # release header / scraped nav
            continue
        if NOISE.search(s):            # media, phone, email, impl. note, urls
            continue
        out.append(s)
    return out


def clean_doc_sentences(sentences):
    """keep_statement_sentences + merge <6-word fragments into the previous
    sentence, WITHIN this document."""
    kept = keep_statement_sentences(sentences)
    merged = []
    for s in kept:
        if len(s.split()) < 6 and merged:
            merged[-1] = merged[-1].rstrip() + ' ' + s
        else:
            merged.append(s)
    return merged


def is_boilerplate_sentence(s: str) -> bool:
    """Final per-sentence guard (belt-and-suspenders)."""
    if not isinstance(s, str) or not s.strip():
        return True
    if END_MARKERS.search(s) or HEADER_MARKERS.search(s) or NOISE.search(s):
        return True
    # name-list heuristic: many separators + mostly Title-case + a roster title.
    seps = s.count(',') + s.count(';')
    tokens = s.split()
    if seps >= 3 and tokens:
        cap_ratio = sum(t[0].isupper() for t in tokens) / len(tokens)
        if cap_ratio > 0.6 and ROSTER.search(s):
            return True
    return False


# ===========================================================================
# 3. Table 1 dictionary (Gorodnichenko et al.) -- filtration + split validity
# ===========================================================================

PANEL_A1 = ["inflation expectation", "interest rate", "bank rate", "fund rate",
            "price", "economic activity", "inflation", "employment"]
PANEL_B1 = ["unemployment", "growth", "exchange rate", "productivity",
            "deficit", "demand", "job market", "monetary policy"]
PANEL_A2 = ["anchor", "cut", "subdue", "decline", "decrease", "reduce", "low",
            "drop", "fall", "fell", "decelerate", "slow", "pause", "pausing",
            "stable", "non-accelerating", "downward", "tighten"]
PANEL_B2 = ["ease", "easing", "rise", "rising", "increase", "expand", "improve",
            "strong", "upward", "raise", "high", "rapid"]
PANEL_C  = ["weren't", "were not", "wasn't", "was not", "did not", "didn't",
            "do not", "don't", "will not", "won't"]

# Word-level matching (not substring), so 'low' never fires on 'below'/'flow'
# and economic jargon isn't matched by accident -- consistent with the table
# enumerating inflections (rise/rising, ease/easing) by hand. Consequence:
# 'expand' won't match 'expanded' and 'price' won't match 'prices'. Set this to
# True to also match inflected forms via a prefix match if recall is too low.
USE_PREFIX_MATCH = False


def _dict_regex(terms, prefix=USE_PREFIX_MATCH):
    """Boundary-anchored alternation over `terms`. Handles multi-word phrases
    ('monetary policy', 'interest rate') and contractions ("don't", 'were not')."""
    parts = [r"\s+".join(re.escape(w) for w in t.split())
             for t in sorted(terms, key=len, reverse=True)]   # longest first
    tail = r"[A-Za-z]*" if prefix else ""
    return re.compile(
        r"(?<![A-Za-z])(?:" + "|".join(parts) + r")" + tail + r"(?![A-Za-z])",
        re.IGNORECASE,
    )


TARGET_RE = _dict_regex(PANEL_A1 + PANEL_B1)                              # filter
TABLE1_RE = _dict_regex(PANEL_A1 + PANEL_B1 + PANEL_A2 + PANEL_B2 + PANEL_C)  # split


def is_target_sentence(s: str) -> bool:
    """Data & Title Filtration: keep iff the sentence has a Panel A1/B1 term."""
    return bool(TARGET_RE.search(s)) if isinstance(s, str) else False


# ===========================================================================
# 4. Connective sentence splitting
# ===========================================================================

SPLIT_RE = re.compile(
    r"\s*(?:;|(?<![A-Za-z])(?:but|however|even\s+though|although|while)(?![A-Za-z]))\s*",
    re.IGNORECASE,
)


def split_on_connectives(sentence: str):
    """Split at the connectives; return the segments only if EVERY segment has a
    Table 1 keyword, otherwise return the sentence unsplit."""
    if not isinstance(sentence, str):
        return []
    segments = [seg.strip() for seg in SPLIT_RE.split(sentence) if seg and seg.strip()]
    if len(segments) <= 1:
        return [sentence.strip()]
    if all(TABLE1_RE.search(seg) for seg in segments):
        return segments
    return [sentence.strip()]


# ===========================================================================
# 5. Pipeline
# ===========================================================================

# Base sentence segmentation. Uses your get_splitter() result `split` if defined,
# otherwise a simple .!? fallback.
try:
    split
except NameError:
    def split(text):
        return re.split(r"(?<=[.!?])\s+", text) if isinstance(text, str) else []

# (a) normalise, then strip the leading wrapper so it can't glue onto sentence 1
docs["clean_text"] = docs["raw_text"].map(clean_text).map(strip_leading_wrapper)

# (b) split, then isolate the statement body per document (original cleaning:
#     stops at the voting record, drops headers/noise, merges short fragments)
docs["sentences"] = docs["clean_text"].map(split).map(clean_doc_sentences)

# (c) explode to one body sentence per row
sents = (
    docs.explode("sentences", ignore_index=True)
        .rename(columns={"sentences": "sentence"})
        .dropna(subset=["sentence"])
)
sents["sentence"] = sents["sentence"].astype(str).str.strip()
sents = sents[sents["sentence"].astype(bool)].reset_index(drop=True)

# (d) final boilerplate guard for anything that slipped through
before = len(sents)
sents = sents[~sents["sentence"].map(is_boilerplate_sentence)].reset_index(drop=True)
print(f"boilerplate guard removed: {before - len(sents)}")

# (e) Data & Title Filtration: keep only target sentences (Panel A1 or B1).
#     Runs AFTER boilerplate removal, so jargon-laden wrapper lines (e.g. the
#     voting record containing 'monetary policy') are already gone.
before = len(sents)
sents = sents[sents["sentence"].map(is_target_sentence)].reset_index(drop=True)
print(f"filtration kept {len(sents)} / {before} sentences (Panel A1/B1 terms)")

# (f) Sentence Splitting at connectives (valid only if every segment is on-topic)
n_pre = len(sents)
sents["sentence"] = sents["sentence"].map(split_on_connectives)
sents = sents.explode("sentence", ignore_index=True)
sents["sentence"] = sents["sentence"].astype(str).str.strip()
sents = sents[sents["sentence"].astype(bool)].reset_index(drop=True)
print(f"splitting: {n_pre} target sentences -> {len(sents)} segments")

# sanity check
sents[["doc_type", "sentence"]].head(10)

boilerplate guard removed: 370
filtration kept 16799 / 40878 sentences (Panel A1/B1 terms)
splitting: 16799 target sentences -> 18019 segments


,doc_type,sentence
0,statement,Economic activity is expanding at a solid pace...
1,statement,Productivity growth and capital investment are...
2,statement,"Job gains have kept pace with the workforce, a..."
3,statement,Inflation remains elevated relative to the Com...
4,statement,The Committee will deliver price stability.
5,statement,Recent indicators suggest that economic activi...
6,statement,"Job gains have remained low, on average, and t..."
7,statement,"Inflation is elevated, in part reflecting the ..."
8,statement,The Committee seeks to achieve maximum employm...
9,statement,The Committee is strongly committed to support...


In [61]:
sents.head(10)

,raw_text,doc_type,clean_text,sentence
0,The Federal Open Market Committee approved the...,statement,The Federal Open Market Committee approved the...,Economic activity is expanding at a solid pace...
1,The Federal Open Market Committee approved the...,statement,The Federal Open Market Committee approved the...,Productivity growth and capital investment are...
2,The Federal Open Market Committee approved the...,statement,The Federal Open Market Committee approved the...,"Job gains have kept pace with the workforce, a..."
3,The Federal Open Market Committee approved the...,statement,The Federal Open Market Committee approved the...,Inflation remains elevated relative to the Com...
4,The Federal Open Market Committee approved the...,statement,The Federal Open Market Committee approved the...,The Committee will deliver price stability.
5,Recent indicators suggest that economic activi...,statement,Recent indicators suggest that economic activi...,Recent indicators suggest that economic activi...
6,Recent indicators suggest that economic activi...,statement,Recent indicators suggest that economic activi...,"Job gains have remained low, on average, and t..."
7,Recent indicators suggest that economic activi...,statement,Recent indicators suggest that economic activi...,"Inflation is elevated, in part reflecting the ..."
8,Recent indicators suggest that economic activi...,statement,Recent indicators suggest that economic activi...,The Committee seeks to achieve maximum employm...
9,Recent indicators suggest that economic activi...,statement,Recent indicators suggest that economic activi...,The Committee is strongly committed to support...


Testing sample break up, Picking at random 500 sentences to test on later, these are kept seperate from all the training of the model to avoid contaminating the data set. 

In [62]:
"""
Pull a random 500-sentence holdout for manual labeling, remove it from the
main `sents` frame, and save both frames.

Assumes `sents` already exists with `doc_type` and `sentence` columns.

Safeguards over a naive version:
  - Pure split function -> does NOT mutate `sents` until the files are saved,
    and the fixed seed makes the split reproducible.
  - Won't overwrite an existing holdout unless OVERWRITE=True, so re-running
    the cell can't silently regenerate a *different* 500 and discard labels
    you've already added.
  - Adds a stable `id` (original row index) and an empty `label` column so the
    CSV is ready to fill in.
  - Warns if OUT_DIR doesn't already exist (catches typos/trailing spaces in
    the path before files land somewhere unexpected).
  - force_ascii=False so en-dashes / curly quotes stay human-readable.
"""

import pandas as pd
from pathlib import Path

# ---- config --------------------------------------------------------------
N_SAMPLE  = 1000
SEED      = 42
DOC_TYPE  = "statement"     # set to None to sample from ALL sentences
OVERWRITE = False           # flip to True only when you want a fresh split

# verify this points at a folder that ALREADY EXISTS (mind trailing spaces / spelling)
OUT_DIR = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Data Folders/"
               "FED BRANCH/POLICY DECISIONS/FOMC ANNOUNCMENTS")

# ---- helper --------------------------------------------------------------
def make_holdout(df, n, seed, mask=None):
    """Return (holdout, remaining) without mutating `df`. Deterministic for a
    given seed; takes all rows if fewer than `n` are available."""
    pool = df if mask is None else df[mask]
    if len(pool) == 0:
        raise ValueError(f"No rows to sample from (doc_type={DOC_TYPE!r}). "
                         f"Check your 'doc_type' values.")
    n_take = min(int(n), len(pool))
    holdout = pool.sample(n=n_take, random_state=seed)
    remaining = df.drop(index=holdout.index)
    return holdout, remaining

# ---- 1. split (no mutation yet) ------------------------------------------
mask = None if DOC_TYPE is None else (sents["doc_type"] == DOC_TYPE)
holdout, remaining = make_holdout(sents, N_SAMPLE, SEED, mask=mask)

# ---- 2. prep the test set for manual labeling ----------------------------
test_df = holdout.copy()
test_df.insert(0, "id", test_df.index)   # original sents index -> trace back later
test_df["label"] = ""                    # <- fill this in by hand
test_df = test_df.reset_index(drop=True)

# ---- 3. paths + guards ---------------------------------------------------
if not OUT_DIR.exists():
    print(f"WARNING: {OUT_DIR} does not exist yet and will be CREATED. "
          f"Double-check the path is what you intended.")
OUT_DIR.mkdir(parents=True, exist_ok=True)

test_csv      = OUT_DIR / "fomc_statements_test_500.csv"
test_jsonl    = OUT_DIR / "fomc_statements_test_500.jsonl"
remaining_csv = OUT_DIR / "fomc_sentences_remaining.csv"

if test_csv.exists() and not OVERWRITE:
    raise FileExistsError(
        f"{test_csv} already exists. Regenerating would create a DIFFERENT random "
        f"sample and discard any labels you've started. Set OVERWRITE=True to proceed."
    )

# ---- 4. save both frames (only mutate `sents` after a successful write) --
test_df.to_csv(test_csv, index=False)
test_df[["id", "sentence", "label"]].to_json(
    test_jsonl, orient="records", lines=True, force_ascii=False
)
remaining.to_csv(remaining_csv, index=False)

sents = remaining.reset_index(drop=True)   # remove the holdout from the main frame

# ---- 5. summary ----------------------------------------------------------
left = (sents["doc_type"] == DOC_TYPE).sum() if DOC_TYPE else len(sents)
print(f"Held out {len(test_df)} sentences for labeling -> {test_csv.name} / {test_jsonl.name}")
print(f"Main frame now {len(sents)} rows ({left} of doc_type={DOC_TYPE!r}) -> {remaining_csv.name}")

Held out 1000 sentences for labeling -> fomc_statements_test_500.csv / fomc_statements_test_500.jsonl
Main frame now 17019 rows (1019 of doc_type='statement') -> fomc_sentences_remaining.csv
